# B-tree Based Library Book Search Demo

This notebook demonstrates a library database indexing system using a B-tree.  
**Book ID** is used as the indexing key for storing and retrieving book records.

## Objectives

This notebook demonstrates how to:

- create a B-tree index for library books
- insert book records into the index
- search for books by Book ID
- observe how node splitting keeps the index balanced


## 1. B-tree Implementation

The following code defines a B-tree using **order `m`**.

In this notebook:

- `m` = maximum number of children in one node
- each node can store up to `m - 1` keys
- if a node becomes full, a split is performed

The main operations shown in this notebook are:

- **search**: find a record by key
- **insert**: add a new record
- **split**: divide a full node when overflow occurs
- **display**: print the tree structure by level


In [1]:
# ------------------------------------------------------
# B-tree node
# Each node stores:
# - leaf: whether the node is a leaf node
# - keys: list of (key, value) pairs
# - children: list of child nodes
# ------------------------------------------------------
class BTreeNode:
    def __init__(self, leaf=True):
        self.leaf = leaf
        self.keys = []
        self.children = []


# ------------------------------------------------------
# B-tree
# m = order of the B-tree
#
# In this notebook:
# - maximum number of children in one node = m
# - maximum number of keys in one node = m - 1
#
# For this demo, m = 4
# so each node can have:
# - up to 4 children
# - up to 3 keys
# ------------------------------------------------------
class BTree:
    def __init__(self, m=4):
        self.root = BTreeNode()
        self.m = m

    # --------------------------------------------------
    # search()
    # Search for a target key in the B-tree.
    # --------------------------------------------------
    def search(self, node, key):
        i = 0

        while i < len(node.keys) and key > node.keys[i][0]:
            i += 1

        if i < len(node.keys) and key == node.keys[i][0]:
            return node.keys[i]

        if node.leaf:
            return None

        return self.search(node.children[i], key)

    # --------------------------------------------------
    # insert()
    # Insert a new key-value record into the B-tree.
    # If the root is full, create a new root and split.
    # --------------------------------------------------
    def insert(self, key, value):
        root = self.root

        if len(root.keys) == self.m - 1:
            new_root = BTreeNode(leaf=False)
            new_root.children.append(root)

            self.split_child(new_root, 0)

            self.root = new_root
            self.insert_non_full(new_root, key, value)
        else:
            self.insert_non_full(root, key, value)

    # --------------------------------------------------
    # insert_non_full()
    # Insert a new key-value record into a node that
    # is guaranteed not to be full.
    # --------------------------------------------------
    def insert_non_full(self, node, key, value):
        i = len(node.keys) - 1

        if node.leaf:
            node.keys.append((None, None))

            while i >= 0 and key < node.keys[i][0]:
                node.keys[i + 1] = node.keys[i]
                i -= 1

            node.keys[i + 1] = (key, value)

        else:
            while i >= 0 and key < node.keys[i][0]:
                i -= 1

            i += 1

            if len(node.children[i].keys) == self.m - 1:
                self.split_child(node, i)

                if key > node.keys[i][0]:
                    i += 1

            self.insert_non_full(node.children[i], key, value)

    # --------------------------------------------------
    # split_child()
    # Split a full child node into two nodes.
    # --------------------------------------------------
    def split_child(self, parent, index):
        node = parent.children[index]

        new_node = BTreeNode(leaf=node.leaf)

        mid = len(node.keys) // 2
        median = node.keys[mid]

        new_node.keys = node.keys[mid + 1:]
        node.keys = node.keys[:mid]

        if not node.leaf:
            new_node.children = node.children[mid + 1:]
            node.children = node.children[:mid + 1]

        parent.children.insert(index + 1, new_node)
        parent.keys.insert(index, median)

    # --------------------------------------------------
    # display()
    # Display the B-tree structure level by level.
    # --------------------------------------------------
    def display(self, node=None, level=0):
        if node is None:
            node = self.root

        print("Level", level, ":", [k[0] for k in node.keys])

        if not node.leaf:
            for child in node.children:
                self.display(child, level + 1)


## 2. Library System Helper Functions

The following functions are used to simulate a simple library indexing system:

- `add_book()` for registering a new book
- `search_book()` for searching the library database
- `show_index()` for displaying the current B-tree index


In [2]:
# ------------------------------------------------------
# add_book()
# Add a new book record to the library B-tree index.
# Book ID is used as the key, and the remaining
# information is stored as a dictionary value.
# ------------------------------------------------------
def add_book(library, book_id, title, author, category):
    book_info = {
        "title": title,
        "author": author,
        "category": category
    }
    print(f"\n[Library System] Adding book record: {book_id} - {title}")
    library.insert(book_id, book_info)


# ------------------------------------------------------
# search_book()
# Search the library B-tree index using Book ID.
# If the record exists, display the full book information.
# Otherwise, print a message indicating that no match was found.
# ------------------------------------------------------
def search_book(library, book_id):
    print(f"\n[Library System] Searching for Book ID {book_id} ...")
    result = library.search(library.root, book_id)

    if result:
        print("[Result] Book found.")
        print("Book ID:", result[0])
        info = result[1]
        print("Title:", info["title"])
        print("Author:", info["author"])
        print("Category:", info["category"])
    else:
        print("[Result] No matching book record found.")


# ------------------------------------------------------
# show_index()
# Display the current B-tree index structure level by level.
# This helps show how the library index changes after
# insertion or node splitting.
# ------------------------------------------------------
def show_index(library):
    print("\n=== Current Library B-tree Index ===")
    library.display()


## 3. Initialize the Library Index and Dataset

In this section, an empty B-tree index is created for the library system.  
The notebook also defines a small sample dataset of book records.

Each record includes:

- Book ID
- Title
- Author
- Category

In this demo, **Book ID** is used as the indexing key in the B-tree.

### Sample Book Records

| Book ID | Title | Author | Category |
|---|---|---|---|
| 1023 | Data Structures | Mark Weiss | CS |
| 2045 | Deep Learning | Ian Goodfellow | AI |
| 3011 | Cell Biology | Alberts | Biology |
| 4020 | Database Systems | Elmasri | CS |
| 5033 | Machine Learning | Tom Mitchell | AI |

In [3]:
# Create an empty B-tree index for the library
library = BTree(m=4)

# Sample book records
books = [
    (1023, "Data Structures", "Mark Weiss", "CS"),
    (2045, "Deep Learning", "Ian Goodfellow", "AI"),
    (3011, "Cell Biology", "Alberts", "Biology"),
    (4020, "Database Systems", "Elmasri", "CS"),
    (5033, "Machine Learning", "Tom Mitchell", "AI")
]

## 4. Register Book Records into the Library Index

The following steps show how the B-tree index changes as book records are inserted one by one.

Each insertion is followed by the updated tree structure so that the effect of sorting and node splitting can be observed clearly.

### 4-1. Insert the first book record

The first book record is inserted into the empty B-tree index.  
Since the root is empty, the key is added without any split.

In [4]:
# Insert 1023

add_book(library, 1023, "Data Structures", "Mark Weiss", "CS")
show_index(library)


[Library System] Adding book record: 1023 - Data Structures

=== Current Library B-tree Index ===
Level 0 : [1023]


### Tree structure after inserting 1023

```text
[1023]
```

The root node contains one key.


### 4-2. Insert the second book record

The second book record is inserted into the same root node.  
The keys remain sorted, and no split is needed.

In [5]:
# Insert 2045

add_book(library, 2045, "Deep Learning", "Ian Goodfellow", "AI")
show_index(library)


[Library System] Adding book record: 2045 - Deep Learning

=== Current Library B-tree Index ===
Level 0 : [1023, 2045]


### Tree structure after inserting 2045

```text
[1023 | 2045]
```

The new key is inserted into the same root node in sorted order.


### 4-3. Insert the third book record

The third book record is inserted into the root node.  
At this point, the root reaches its maximum number of keys.

In [6]:
# Insert 3011

add_book(library, 3011, "Cell Biology", "Alberts", "Biology")
show_index(library)


[Library System] Adding book record: 3011 - Cell Biology

=== Current Library B-tree Index ===
Level 0 : [1023, 2045, 3011]


### Tree structure after inserting 3011

```text
[1023 | 2045 | 3011]
```

At this point, the root node is full because it already contains the maximum number of keys.


### 4-4. Insert the fourth book record

A new book record is inserted when the root is already full.  
The root node must be split before the insertion can continue.

In [7]:
# Insert 4020

add_book(library, 4020, "Database Systems", "Elmasri", "CS")
show_index(library)


[Library System] Adding book record: 4020 - Database Systems

=== Current Library B-tree Index ===
Level 0 : [2045]
Level 1 : [1023]
Level 1 : [3011, 4020]


### Tree structure after inserting 4020

```text
        [2045]
       /      \
   [1023]   [3011 | 4020]
```

A split occurs because the original root node is full.  
The median key `2045` moves up and becomes the new root.


### 4-5. Insert the fifth book record

The next book record is inserted into the right child of the root.  
This child still has space, so the key is added without another split.

In [8]:
# Insert 5033

add_book(library, 5033, "Machine Learning", "Tom Mitchell", "AI")
show_index(library)


[Library System] Adding book record: 5033 - Machine Learning

=== Current Library B-tree Index ===
Level 0 : [2045]
Level 1 : [1023]
Level 1 : [3011, 4020, 5033]


### Tree structure after inserting 5033

```text
        [2045]
       /      \
   [1023]   [3011 | 4020 | 5033]
```

The new key is inserted into the right child, which is still allowed to hold up to three keys.


## 5. Add a New Book and Observe Index Update

A new book record is added to the library system.  
This step shows how the B-tree index is updated when a full node must be split.


In [9]:
# Insert 6010

add_book(library, 6010, "Artificial Intelligence", "Russell", "AI")
show_index(library)


[Library System] Adding book record: 6010 - Artificial Intelligence

=== Current Library B-tree Index ===
Level 0 : [2045, 4020]
Level 1 : [1023]
Level 1 : [3011]
Level 1 : [5033, 6010]


### Tree structure after inserting 6010

Before inserting `6010`, the right child was already full:

```text
        [2045]
       /      \
   [1023]   [3011 | 4020 | 5033]
```

After inserting `6010`, another split occurs:

```text
          [2045 | 4020]
         /      |      \
    [1023]   [3011]   [5033 | 6010]
```

The median key `4020` moves up to the root, and the full child node is divided into two smaller nodes.


## 6. Search for Books by Book ID

The following examples show how B-tree search works.

A search starts from the root node, compares the target key with the keys in that node, and then moves to the appropriate child node if needed.

The examples below include:

- a successful search case
- an unsuccessful search case

In [10]:
# Search for an existing book
search_book(library, 3011)

# Search for a non-existing book
search_book(library, 9999)


[Library System] Searching for Book ID 3011 ...
[Result] Book found.
Book ID: 3011
Title: Cell Biology
Author: Alberts
Category: Biology

[Library System] Searching for Book ID 9999 ...
[Result] No matching book record found.


### Search path for Book ID 3011

The current tree structure is:

```text
          [2045 | 4020]
         /      |      \
    [1023]   [3011]   [5033 | 6010]
```

To search for `3011`:

1. Start at the root node [2045 | 4020]
2. Compare `3011` with the keys in the root
3. Since `3011` is between `2045` and `4020`, move to the middle child
4. The key `3011` is found in [3011]

This search example shows that the B-tree can locate records efficiently through a balanced structure.

## 7. Summary

This notebook simulates a library database indexing system using a B-tree.

It demonstrates:

- how book records are stored in a B-tree index
- how books can be searched efficiently by Book ID
- how node splitting helps maintain a balanced structure as new records are added

This example illustrates why B-trees are useful in record retrieval systems such as library databases.
